# B6i · CardioGenAI: re-engineering a drug for less hERG block
**Series B: Deep Learning for Molecules, Post 6i** &nbsp;|&nbsp; companion *primer* notebook

CardioGenAI (Kyro, Martin, Watt and Batista, *J. Cheminform.* 2025) re-engineers a drug that blocks the hERG potassium channel. An **autoregressive transformer** trained on about $5.5$ million SMILES strings generates molecules *conditioned on the scaffold and ten physicochemical properties* of the input drug; **discriminative models** (a bidirectional-transformer feature vector, a fingerprint and a graph network) filter the generated molecules by predicted hERG, NaV1.5 and CaV1.2 pIC$_{50}$; and the survivors are ranked by the **cosine similarity** of RDKit descriptor vectors, from which redundant descriptors have been removed by **mutual information**. The paper's headline: among $100$ refined candidates for pimozide is fluspirilene, of the same class, with a hERG pIC$_{50}$ of $5.638$ against $8.520$.

This primer works with **what can be rebuilt and checked without the trained models**, with NumPy only: the potency arithmetic behind the claims (thresholds, fold changes, the $700$-fold statement); the **confusion matrices behind Tables 1 and 2**, recovered from the six reported metrics; a check of Table 3 (predicted against experimental pIC$_{50}$); a SMILES parser, **Bemis--Murcko scaffolds** and properties (pimozide and fluspirilene have *different* scaffolds); mutual-information pruning and cosine similarity (and a pitfall); and **CardioGenAI in miniature**: a transformer with the paper's *summed* conditioning (token, position, scaffold and property embeddings), trained by hand-written backpropagation on a combinatorial corpus, evaluated for validity, scaffold retention and property control, and run through a toy filter-and-rank pipeline. Part II runs the real discriminative models of the repository on pimozide and fluspirilene, and checks the primer's functions against RDKit and scikit-learn.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in about a minute on a CPU (the miniature transformer is trained from scratch). **Part II** needs scikit-learn, RDKit and the repository (it clones it and installs what it can); a GPU is not required for the inference of Part II.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | pIC$_{50}$, thresholds, and the fold changes of the paper |
| 2 | the confusion matrices behind Tables 1 and 2 |
| 3 | Table 3: predicted against experimental pIC$_{50}$ |
| 4 | graphs, properties and scaffolds of pimozide and fluspirilene |
| 5 | mutual-information pruning and cosine similarity |
| 6 | the generator in miniature: summed conditioning |
| 7 | the framework in miniature: generate, filter, rank |
| **Part II** | the real discriminative models; RDKit and scikit-learn checks |

## 0. Setup

In [ ]:
import math, time, itertools, importlib.util, os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

## 1. Potency arithmetic
The paper labels a compound a **hERG blocker** when its $\mathrm{pIC}_{50}=-\log_{10}(\mathrm{IC}_{50}/\mathrm{M})\ge5.0$ (an IC$_{50}$ of $10\,\mu$M or lower); a drop of one pIC$_{50}$ unit is a tenfold weaker binding. Everything the paper says about potency is this arithmetic.
**Stop and predict:** pimozide has an experimental pIC$_{50}$ of $8.520$ and fluspirilene $5.638$. How many times weaker is fluspirilene, and how many times weaker do the *predicted* values ($7.629$ and $5.785$) say it is?

In [ ]:
pic50 = lambda ic50_nM: -math.log10(ic50_nM * 1e-9); ic50_nM = lambda p: 10 ** (-p) * 1e9
pim_exp, flu_exp, pim_pred, flu_pred = 8.520, 5.638, 7.629, 5.785
print(f"threshold 5.0 = IC50 of {ic50_nM(5.0) / 1000:.0f} uM; 6.0 (the filter used for pimozide) = {ic50_nM(6.0) / 1000:.0f} uM; 7.0 = {ic50_nM(7.0):.0f} nM")
print(f"pimozide experimental 8.520 = IC50 {ic50_nM(pim_exp):.2f} nM; fluspirilene 5.638 = IC50 {ic50_nM(flu_exp) / 1000:.2f} uM")
fold_exp, fold_pred = 10 ** (pim_exp - flu_exp), 10 ** (pim_pred - flu_pred)
print(f"fold change, experimental values: 10^({pim_exp - flu_exp:.3f}) = {fold_exp:.0f}-fold (the paper: 'over 700-fold') | predicted values: 10^({pim_pred - flu_pred:.3f}) = {fold_pred:.0f}-fold")
print(f"pimozide: the paper also quotes an IC50 of about 18 nM from the literature: pIC50 = {pic50(18):.3f}, which differs from 8.520 by {pim_exp - pic50(18):.2f} log units, a factor of {10 ** (pim_exp - pic50(18)):.1f}; the paper notes that assay conditions alone can cause variations above 0.5 log units")
print(f"error of the regression model for pimozide: {pim_exp - pim_pred:.3f} (the paper: 0.891) | for fluspirilene: {flu_exp - flu_pred:+.3f}")
print("a candidate is kept if its predicted pIC50 is below 6.0; for the 100 candidates the paper reports 4.64 to 6.00 with a mean of 5.59")
assert abs(fold_exp - 762) < 1 and abs(fold_pred - 70) < 1 and abs(pim_exp - pim_pred - 0.891) < 1e-9 and 0.7 < pim_exp - pic50(18) < 0.8

## 2. The confusion matrices behind Tables 1 and 2
Table 1 (hERG) and Table 2 (NaV1.5, CaV1.2) report six numbers per model: accuracy, sensitivity, specificity, F1, correct classification rate (CCR) and the Matthews correlation coefficient (MCC), in percent, on a **test set of fixed size** ($474$, $142$ and $81$ compounds). A confusion matrix of integers $(TP,FN,TN,FP)$ that adds up to the test-set size and reproduces all six rounded numbers is very constrained, so we can **search for it**. If the matrix is unique, the table is internally consistent and we learn the class balance; if the rows of a table disagree on the class balance, they were not computed on the same compounds.
**Stop and predict:** for the CaV1.2 test set of $81$ compounds, by how many compounds do CardioGenAI ($91.4\%$) and CToxPred-Cav ($86.4\%$) differ?

In [ ]:
def metrics6(tp, fn, tn, fp):
    n = tp + fn + tn + fp; sn, sp = tp / (tp + fn), tn / (tn + fp); den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.array([(tp + tn) / n, sn, sp, tp / (tp + 0.5 * (fp + fn)), (sn + sp) / 2, (tp * tn - fp * fn) / den]) * 100
def reconstruct(n, rep, tol=0.0501):
    """all integer (TP, FN, TN, FP) with sum n whose six metrics round to the reported values (AC, SN, SP, F1, CCR, MCC)"""
    rep = np.asarray(rep, float); lo, hi = n, 0
    for dac, dsn, dsp in itertools.product((-5e-4, 5e-4), repeat=3):
        a, s1, s2 = rep[0] / 100 + dac, rep[1] / 100 + dsn, rep[2] / 100 + dsp
        if abs(s1 - s2) > 1e-6: p = (a - s2) / (s1 - s2) * n; lo, hi = min(lo, p), max(hi, p)
    out = []
    for P in range(max(1, int(lo) - 2), min(n - 1, int(hi) + 3)):
        N = n - P; tp = np.arange(P + 1)[:, None].astype(float); tn = np.arange(N + 1)[None, :].astype(float); fn, fp = P - tp, N - tn
        ac, sn, sp = (tp + tn) / n * 100, tp / P * 100, tn / N * 100; f1 = tp / (tp + 0.5 * (fp + fn)) * 100; ccr = (sn + sp) / 2; den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        mcc = np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0) * 100
        ok = (np.abs(ac - rep[0]) <= tol) & (np.abs(sn - rep[1]) <= tol) & (np.abs(sp - rep[2]) <= tol) & (np.abs(f1 - rep[3]) <= tol) & (np.abs(ccr - rep[4]) <= tol + 1e-4) & (np.abs(mcc - rep[5]) <= tol)
        out += [(int(i), int(P - i), int(j), int(N - j)) for i, j in zip(*np.where(ok))]
    return out
T2 = {("NaV1.5", "CardioGenAI"): (142, (89.4, 95.9, 75.6, 92.5, 85.7, 75.1)), ("NaV1.5", "CToxPred-Nav"): (142, (81.7, 85.6, 73.3, 86.5, 79.5, 58.2)), ("CaV1.2", "CardioGenAI"): (81, (91.4, 96.2, 82.8, 93.5, 89.5, 81.0)), ("CaV1.2", "CToxPred-Cav"): (81, (86.4, 96.2, 69.0, 90.1, 82.6, 70.2))}
rows = []
for (ch, model), (n, rep) in T2.items():
    s = reconstruct(n, rep); rows.append((ch, model, n, len(s), *(s[0] if len(s) == 1 else (None,) * 4)))
t2 = pd.DataFrame(rows, columns=["channel", "model", "test compounds", "integer solutions", "TP", "FN", "TN", "FP"]); print(t2.to_string(index=False))
print("\nevery row has a unique matrix; the two models of a channel have the same numbers of blockers and non-blockers:", {ch: (int(t2[t2.channel == ch].iloc[0][["TP", "FN"]].sum()), int(t2[t2.channel == ch].iloc[0][["TN", "FP"]].sum())) for ch in ("NaV1.5", "CaV1.2")})
r_ = t2.iloc[[0, 1]]; c_ = t2.iloc[[2, 3]]; print(f"NaV1.5: CardioGenAI classifies {int(r_.iloc[0].TP + r_.iloc[0].TN)} compounds correctly, CToxPred-Nav {int(r_.iloc[1].TP + r_.iloc[1].TN)}; CaV1.2: {int(c_.iloc[0].TP + c_.iloc[0].TN)} against {int(c_.iloc[1].TP + c_.iloc[1].TN)}: a difference of {int(c_.iloc[0].TP + c_.iloc[0].TN - c_.iloc[1].TP - c_.iloc[1].TN)} compounds of 81")
assert (t2["integer solutions"] == 1).all() and tuple(t2.iloc[0][["TP", "FN", "TN", "FP"]].astype(int)) == (93, 4, 34, 11) and tuple(t2.iloc[2][["TP", "FN", "TN", "FP"]].astype(int)) == (50, 2, 24, 5)

In [ ]:
T1 = {"CardioGenAI": (83.5, 86.2, 80.3, 85.1, 83.2, 66.7), "CToxPred-hERG": (81.4, 86.7, 74.6, 83.9, 80.7, 62.1), "CardioTox": (81.2, 83.0, 78.9, 83.1, 81.0, 61.9), "ADMETlab 2.0": (71.7, 71.6, 71.8, 73.8, 71.7, 43.1), "ADMETsar 2.0": (68.5, 84.5, 48.3, 75.0, 66.4, 35.5), "CardPred": (56.1, 52.7, 60.3, 57.0, 56.5, 13.0)}
t0 = time.perf_counter(); rows = []
for name, rep in T1.items():
    s474 = reconstruct(474, rep); s473 = reconstruct(473, rep); rows.append((name, len(s474), len(s473), s473[0] if len(s473) == 1 else None))
print(pd.DataFrame(rows, columns=["model", "solutions for 474 compounds (all six metrics)", "solutions for 473 compounds", "matrix at 473 (TP, FN, TN, FP)"]).to_string(index=False), f"[{time.perf_counter() - t0:.0f} s]")
# CardioGenAI: accuracy, sensitivity and specificity alone fix a unique matrix at 474
def three_only(n, rep, tol=0.0501):
    out = []
    for P in range(1, n):
        N = n - P; tp = np.arange(P + 1)[:, None].astype(float); tn = np.arange(N + 1)[None, :].astype(float); ok = (np.abs((tp + tn) / n * 100 - rep[0]) <= tol) & (np.abs(tp / P * 100 - rep[1]) <= tol) & (np.abs(tn / N * 100 - rep[2]) <= tol)
        out += [(int(i), int(P - i), int(j), int(N - j)) for i, j in zip(*np.where(ok))]
    return out
cg = three_only(474, T1["CardioGenAI"]); print("\nCardioGenAI, accuracy + sensitivity + specificity only, 474 compounds:", cg, "-> implied metrics", metrics6(*cg[0]).round(2), "| reported", T1["CardioGenAI"])
print("the implied F1 is 85.23, the reported 85.1; CCR 83.24 and MCC 66.68 agree with the reported 83.2 and 66.7")
common = [s[0] for s in (reconstruct(473, T1[m]) for m in ("CToxPred-hERG", "CardioTox", "ADMETlab 2.0", "ADMETsar 2.0"))]
print("four baselines at 473 compounds: numbers of blockers and non-blockers:", [(tp + fn, tn + fp) for tp, fn, tn, fp in common])
assert len(cg) == 1 and cg[0] == (225, 36, 171, 42) and len({(tp + fn, tn + fp) for tp, fn, tn, fp in common}) == 1 and all(len(reconstruct(474, T1[m])) == 0 for m in T1)

## 3. Table 3: predicted against experimental pIC$_{50}$
Table 3 lists the $11$ FDA-approved drugs that the hERG regression model predicts above $7.0$, with the experimental value from the benchmark; $8$ of them were in the training set. The paper points out that for pimozide, astemizole and dofetilide the prediction is about an order of magnitude below the experiment, and attributes it to *regression toward the mean* for compounds with very high pIC$_{50}$. We compute the statistics of the table.
**Stop and predict:** is the model biased low or high on these $11$ drugs, and what does selecting drugs by a *predicted* value above $7.0$ do to the bias?

In [ ]:
t3 = pd.DataFrame([("Nintedanib", 8.234, 8.585, True), ("Ibutilide", 7.977, 8.000, True), ("Pimozide", 7.629, 8.520, True), ("Halofantrine", 7.588, 7.398, False), ("Astemizole", 7.562, 8.538, True), ("Tolterodine", 7.311, 7.886, False),
                   ("Droperidol", 7.300, 7.495, True), ("Dofetilide", 7.164, 8.194, True), ("Haloperidol decanoate", 7.149, 6.921, False), ("Amiodarone", 7.127, 7.523, True), ("Terfenadine", 7.022, 7.252, True)], columns=["drug", "predicted", "experimental", "in training set"])
t3["error"] = t3.predicted - t3.experimental; print(t3.round(3).to_string(index=False))
e = t3.error.values; print(f"\nmean error {e.mean():+.3f}, RMSE {math.sqrt((e ** 2).mean()):.3f}, MAE {np.abs(e).mean():.3f}; {int((e < 0).sum())} of 11 predictions are below the experiment")
tr_, nt_ = t3[t3["in training set"]], t3[~t3["in training set"]]; print(f"in the training set (8): mean error {tr_.error.mean():+.3f}, RMSE {math.sqrt((tr_.error ** 2).mean()):.3f} | not in the training set (3): mean error {nt_.error.mean():+.3f}, RMSE {math.sqrt((nt_.error ** 2).mean()):.3f}")
slope, icpt = np.polyfit(t3.experimental, t3.predicted, 1); print(f"least-squares line of predicted against experimental: slope {slope:.2f} (1.0 would be unbiased), correlation {np.corrcoef(t3.predicted, t3.experimental)[0, 1]:.2f}")
big = t3[t3.error.abs() > 0.85]; print("errors of about an order of magnitude (|error| > 0.85):", dict(zip(big.drug, big.error.round(3))), "| their experimental values are among the", sorted(t3.experimental, reverse=True).index(big.experimental.min()) + 1, "highest of the 11")
print(f"if the drugs were selected by a predicted value above 7.0, noise in the prediction would make predicted values too HIGH on average; here the mean error is {e.mean():+.2f}, so the underestimation of the high experimental values dominates")
assert abs(e.mean() + 0.386) < 5e-4 and (e < 0).sum() == 9 and slope < 0.5 and abs(tr_.error.mean() + 0.512) < 5e-3

## 4. Graphs, properties and scaffolds
The generator is conditioned on the **Bemis--Murcko scaffold** of the input molecule and ten properties. The scaffold keeps the ring systems and the linkers between them and removes the side chains; atoms double-bonded to the scaffold are kept, other terminal atoms are removed one layer at a time until nothing more can go. We parse SMILES into a graph (the parser of Post~A6b), compute the scaffold, and compare two graphs by a Weisfeiler--Lehman hash of the scaffold subgraph (an invariant: equal graphs give equal hashes). Five of the ten properties follow directly from the graph: molecular weight, number of rings, rotatable bonds (here: single non-ring bonds between two atoms that each have another neighbor), hydrogen-bond donors (N--H and O--H) and the number of heteroatoms.
**Stop and predict:** do pimozide and fluspirilene have the same scaffold?

In [ ]:
import re
MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "Br": 79.904}
Z = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "S": 16, "Cl": 17, "Br": 35}
VAL = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1, "Br": 1}
TOKEN = re.compile(r"\[[^\]]+\]|Cl|Br|[BCNOSPFI]|[bcnosp]|\(|\)|=|#|-|:|\/|\\|\d|%\d\d|\.")
def parse_smiles(smi):
    """SMILES -> atoms (element, aromatic, charge, explicit_H or None) and bonds (i, j, order); order 1.5 marks an aromatic bond"""
    atoms, bonds, stack, prev, pending, rings = [], [], [], None, None, {}
    for tok in TOKEN.findall(smi):
        if tok == "(": stack.append(prev)
        elif tok == ")": prev = stack.pop()
        elif tok in "-=#:": pending = {"-": 1, "=": 2, "#": 3, ":": 1.5}[tok]
        elif tok.isdigit():
            if tok in rings: j, o = rings.pop(tok); bonds.append((prev, j, o if o is not None else (pending or (1.5 if atoms[prev][1] and atoms[j][1] else 1)))); pending = None
            else: rings[tok] = (prev, pending); pending = None
        else:
            if tok.startswith("["):
                m = re.match(r"\[([A-Za-z][a-z]?)(H\d?)?([+-]\d?)?\]", tok); el = m.group(1); arom = el.islower(); el = el.capitalize()
                nh = 0 if not m.group(2) else (1 if m.group(2) == "H" else int(m.group(2)[1:])); ch = 0 if not m.group(3) else (int(m.group(3)[1:] or 1) * (1 if m.group(3)[0] == "+" else -1)); atoms.append((el, arom, ch, nh))
            else: arom = tok.islower(); atoms.append((tok.capitalize(), arom, 0, None))
            i = len(atoms) - 1
            if prev is not None: bonds.append((prev, i, pending if pending else (1.5 if atoms[prev][1] and arom else 1)))
            prev, pending = i, None
    return atoms, bonds
def add_hydrogens(atoms, bonds):
    """implicit hydrogens from the default valences (aromatic bonds count 1, plus one pi bond for carbon and pyridine-type nitrogen)"""
    used = [0.0] * len(atoms); nb = [[] for _ in atoms]
    for i, j, o in bonds: nb[i].append((j, o)); nb[j].append((i, o)); w = 1 if o == 1.5 else o; used[i] += w; used[j] += w
    H = []
    for i, (el, arom, ch, nh) in enumerate(atoms):
        if nh is not None: H.append(nh); continue                                             # bracket atoms: the hydrogens are written
        v = VAL[el] + (1 if (el == "N" and ch == 1) else 0) - (1 if (el == "O" and ch == 1) else 0)
        u = used[i]
        if arom and el in ("C", "N") and u < v: u += 1                                          # the pi bond of an aromatic atom
        H.append(int(max(0, v - u)))
    return H, nb
def ring_atoms(n, bonds):
    """atoms and bonds that lie on a ring: a bond is a ring bond if its ends stay connected without it"""
    adj = {i: [] for i in range(n)}
    for k, (i, j, o) in enumerate(bonds): adj[i].append((j, k)); adj[j].append((i, k))
    def connected(a, b, skip):
        seen, st = {a}, [a]
        while st:
            x = st.pop()
            if x == b: return True
            for y, k in adj[x]:
                if k != skip and y not in seen: seen.add(y); st.append(y)
        return False
    rb = [connected(i, j, k) for k, (i, j, o) in enumerate(bonds)]; ra = set()
    for k, (i, j, o) in enumerate(bonds):
        if rb[k]: ra.update((i, j))
    return ra, rb
def mol_weight(atoms, H): return sum(MASS[a[0]] for a in atoms) + sum(H) * MASS["H"]

In [ ]:
def murcko_atoms(smi):
    at, bd = parse_smiles(smi); n = len(at); keep = set(range(n)); nb_ = {i: {} for i in range(n)}
    for i, j, o in bd: nb_[i][j] = o; nb_[j][i] = o
    ra, _ = ring_atoms(n, bd)
    while True:
        rem = [i for i in keep if i not in ra and sum(1 for j in nb_[i] if j in keep) <= 1 and all(o != 2 for j, o in nb_[i].items() if j in keep)]          # terminal atoms that are not double-bonded to the scaffold
        if not rem: break
        keep -= set(rem)
    return at, bd, sorted(keep)
def wl_hash(at, bd, atoms, rounds=3):
    s = set(atoms); nb_ = {i: [] for i in s}
    for i, j, o in bd:
        if i in s and j in s: nb_[i].append((j, o)); nb_[j].append((i, o))
    lab = {i: (at[i][0], at[i][1], len(nb_[i])) for i in s}
    for _ in range(rounds): lab = {i: hash((lab[i], tuple(sorted((o, lab[j]) for j, o in nb_[i])))) for i in s}
    return hash(tuple(sorted(lab.values()))) if s else 0
def scaffold_key(smi): at, bd, keep = murcko_atoms(smi); return (len(keep), wl_hash(at, bd, keep))
def props(smi):
    at, bd = parse_smiles(smi); H, nb_ = add_hydrogens(at, bd); n = len(at); ra, rb = ring_atoms(n, bd); deg = [len(nb_[i]) for i in range(n)]
    rot = sum(1 for k, (i, j, o) in enumerate(bd) if o == 1 and not rb[k] and deg[i] > 1 and deg[j] > 1); hbd = sum(1 for i in range(n) if at[i][0] in ("N", "O") and H[i] > 0)
    return np.array([mol_weight(at, H), len(bd) - n + 1, rot, hbd, sum(a[0] != "C" for a in at)], float)
PROP_NAMES = ["MW", "rings", "rotatable bonds", "HBD", "heteroatoms"]
pim = "O=C1Nc2ccccc2N1C1CCN(CCCC(c2ccc(F)cc2)c2ccc(F)cc2)CC1"; flu = "O=C1NCN(c2ccccc2)C13CCN(CCCC(c2ccc(F)cc2)c2ccc(F)cc2)CC3"
rows = []
for nm, s in (("pimozide", pim), ("fluspirilene", flu)):
    at, bd, keep = murcko_atoms(s); rows.append((nm, len(at), len(keep), *props(s)))
print(pd.DataFrame(rows, columns=["molecule", "heavy atoms", "scaffold atoms"] + PROP_NAMES).round(2).to_string(index=False))
print("same Murcko scaffold:", scaffold_key(pim) == scaffold_key(flu), "| pimozide: benzimidazolone (fused, N-substituted) - piperidine - propyl linker - bis(4-fluorophenyl)methyl; fluspirilene: a spiro imidazolidinone with an N-phenyl group instead")
print("molecular weights 461.56 and 475.58 g/mol agree with the values of the literature for pimozide and fluspirilene (461.55, 475.57)")
assert abs(props(pim)[0] - 461.56) < 0.02 and abs(props(flu)[0] - 475.58) < 0.02 and scaffold_key(pim) != scaffold_key(flu) and murcko_atoms(pim)[2].__len__() == 32

## 5. Mutual-information pruning and cosine similarity
After filtering, the candidates are ranked by the cosine similarity between the descriptor vector of the input and that of each candidate; before that, redundant descriptors are removed using the **mutual information** between pairs of descriptors, $I(X;Y)=\sum p(x,y)\ln\frac{p(x,y)}{p(x)p(y)}$ (estimated here from histograms with $8$ bins; the repository's exact threshold is not stated in the paper, so ours is an illustration: drop a descriptor whose normalized mutual information $I/\min(H_X,H_Y)$ with an earlier one exceeds $0.8$). A pitfall deserves a test: the **cosine similarity of unscaled descriptor vectors** is dominated by the descriptors with the largest numerical values.
**Stop and predict:** the cosine similarity of the *raw* descriptor vectors of two very different molecules (benzene and an octyl chain with a hydroxyl group): near $0$ or near $1$?

In [ ]:
SCAF = {"benzene": "c1c{A}cc{B}cc1", "pyridine": "n1cc{A}cc{B}c1", "cyclohexane": "C1CC{A}CC{B}C1", "piperidine": "N({B})1CCC{A}CC1", "naphthalene": "c1cc{A}c2ccccc2c1", "biphenyl": "c1cc{A}ccc1-c1ccc{B}cc1",
        "phenethylpiperidine": "c1cc{A}ccc1CCN1CCCC{B}C1", "diphenylmethane": "c1ccccc1Cc1ccc{A}cc1", "indole": "c1cc{A}c2[nH]ccc2c1", "cyclopentane": "C1CCC{A}C1"}
SUB = ["", "F", "Cl", "Br", "C", "CC", "CCC", "OC", "O", "N", "C(F)(F)F", "C#N", "CO", "CN", "OCC", "C(C)C"]
build = lambda name, a, b: SCAF[name].replace("{A}", f"({a})" if a else "").replace("{B}", f"({b})" if b else "")
scaffold_smiles = lambda name: SCAF[name].replace("{A}", "").replace("{B}", "")
seen, corpus = set(), []
for name in SCAF:
    for a, b in itertools.product(SUB, SUB):
        s = build(name, a, b)
        if s not in seen: seen.add(s); corpus.append((s, name))
smi_all = [s for s, _ in corpus]; fam = [n for _, n in corpus]
mis = sum(scaffold_key(s) != scaffold_key(scaffold_smiles(n)) for s, n in corpus)
print(f"a combinatorial corpus: {len(corpus)} molecules from {len(SCAF)} scaffolds and {len(SUB)} substituents; molecules whose Murcko scaffold differs from the template: {mis}")
def desc12(smi):
    at, bd = parse_smiles(smi); H, nb_ = add_hydrogens(at, bd); el = [a[0] for a in at]; n = len(at); p5 = props(smi)
    return np.array([*p5, n, el.count("C"), el.count("N"), el.count("O"), sum(e in ("F", "Cl", "Br") for e in el), sum(a[1] for a in at), sum(H)], float)
D12 = ["MW", "rings", "rotatable bonds", "HBD", "heteroatoms", "heavy atoms", "C", "N", "O", "halogens", "aromatic atoms", "hydrogens"]; Dm = np.array([desc12(s) for s in smi_all])
def entropy(l): p = np.bincount(l) / len(l); p = p[p > 0]; return float(-(p * np.log(p)).sum())
def mutual_info(a, b):
    M = np.zeros((a.max() + 1, b.max() + 1))
    for i, j in zip(a, b): M[i, j] += 1
    p = M / M.sum(); px, py = p.sum(1, keepdims=True), p.sum(0, keepdims=True); nz = p > 0; return float((p[nz] * np.log(p[nz] / (px @ py)[nz])).sum())
bins = np.stack([np.digitize(Dm[:, j], np.quantile(Dm[:, j], np.linspace(0, 1, 9)[1:-1])) if len(np.unique(Dm[:, j])) > 8 else np.unique(Dm[:, j], return_inverse=True)[1] for j in range(Dm.shape[1])], 1)
MI = np.array([[mutual_info(bins[:, i], bins[:, j]) for j in range(12)] for i in range(12)]); H_ = [entropy(bins[:, j]) for j in range(12)]
norms = {"I / min(Hx, Hy)": lambda i, j: MI[i, j] / min(H_[i], H_[j]), "2 I / (Hx + Hy)": lambda i, j: 2 * MI[i, j] / (H_[i] + H_[j]), "I / sqrt(Hx Hy)": lambda i, j: MI[i, j] / math.sqrt(H_[i] * H_[j])}
def prune(f, thr):
    kept, rem = [], {}
    for j in range(12):
        hit = [i for i in kept if f(i, j) > thr]
        if hit: rem[D12[j]] = D12[hit[0]]
        else: kept.append(j)
    return kept, rem
rows = []
for nm, f in norms.items():
    for thr in (0.6, 0.7, 0.8): kept, rem = prune(f, thr); rows.append((nm, thr, len(kept), "; ".join(f"{a} (~{b})" for a, b in rem.items()) or "none"))
print("the entropies of the binned descriptors (nats):", dict(zip(D12, np.round(H_, 2)))); print(pd.DataFrame(rows, columns=["normalization", "threshold", "descriptors kept (of 12)", "removed (redundant with)"]).to_string(index=False))
print("the surviving descriptors depend on the normalization and on the threshold; the paper states neither, so a reproduction has to choose")
assert len({r[3] for r in rows}) >= 3

In [ ]:
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
benz, octol = desc12("c1ccccc1"), desc12("CCCCCCCCO"); mu, sd = Dm.mean(0), Dm.std(0); z = lambda v: (v - mu) / sd
print(f"raw descriptor vectors: cosine similarity of benzene and octanol = {cos(benz, octol):.3f}; after standardizing the columns = {cos(z(benz), z(octol)):.3f}")
inp = "c1cc(Cl)ccc1CCN1CCCCC1"; v_in = desc12(inp); raw = np.array([cos(v_in, v) for v in Dm]); st = np.array([cos(z(v_in), z(v)) for v in Dm])
print(f"input {inp}: over the {len(Dm)} molecules of the corpus the raw cosine similarity ranges from {raw.min():.3f} to {raw.max():.3f} (nearly everything looks similar); the standardized one from {st.min():.2f} to {st.max():.2f}")
top = np.argsort(-st)[:5]; print("most similar by the standardized cosine:", [(smi_all[i], round(float(st[i]), 3)) for i in top])
print("the raw similarity is dominated by the molecular weight (values of 100 to 300 against 0 to 3 for the counts), the standardized one weights all descriptors alike; the paper does not state which scaling the code applies, so check it before relying on the ranking")
assert raw.min() > 0.9 and st.min() < 0.0 and cos(benz, octol) > 0.9 and cos(z(benz), z(octol)) < 0.9

## 6. The generator in miniature: summed conditioning
The paper's autoregressive transformer receives, at every position of a SMILES sequence, the **sum** of four embeddings: the token, its position, the *scaffold* token at the same position (the scaffold's SMILES, padded), and a linear map of the property vector; eight blocks and eight heads follow. We build the same input scheme at a small scale and train it by hand-written backpropagation: embedding width $48$, one block with one attention head and a feed-forward layer, $5$ properties, a corpus of $1{,}600$ molecules (we hold out $200$ and condition on their scaffolds and properties). We cannot train on $5.5$ million molecules, but the measurements are those of the paper: validity, novelty, scaffold retention and **property control**.
**Stop and predict:** if the scaffold and the properties of a held-out molecule are given, how often is the sampled SMILES valid, and how often is it that very molecule?

In [ ]:
def softmax(x, axis=-1): x = x - x.max(axis, keepdims=True); e = np.exp(x); return e / e.sum(axis, keepdims=True)
class Vocab:
    def __init__(self, smiles_list): toks = sorted({t for s in smiles_list for t in TOKEN.findall(s)}); self.itos = ["<pad>", "<cls>", "<eos>"] + toks; self.stoi = {t: i for i, t in enumerate(self.itos)}
    def enc(self, smi, T, eos=True): ids = [self.stoi[t] for t in TOKEN.findall(smi)] + ([2] if eos else []); return ids + [0] * (T - len(ids))
def arom_cycle(i, adj, size):
    """is atom i on a cycle of the given size made only of aromatic bonds?"""
    def dfs(path):
        if len(path) == size: return i in adj[path[-1]]
        return any(dfs(path + [j]) for j in adj[path[-1]] if j not in path)
    return dfs([i])
def valid(smi):
    """syntax, ring closures, branches (no empty branch), the valence of aliphatic atoms, and every aromatic atom on an all-aromatic 5- or 6-ring (Part II compares with RDKit)"""
    try:
        if not smi or "()" in smi or smi.count("(") != smi.count(")") or smi.count("[") != smi.count("]") or "".join(TOKEN.findall(smi)) != smi or smi[0] in "()=#-" or smi[-1] in "(=#-": return False
        depth = 0
        for t in TOKEN.findall(smi):
            depth += (t == "(") - (t == ")")
            if depth < 0: return False
        rings = {}
        for t in TOKEN.findall(smi):
            if t.isdigit(): rings[t] = rings.get(t, 0) + 1
        if any(v % 2 for v in rings.values()): return False
        at, bd = parse_smiles(smi); used = [0.0] * len(at); deg = [0] * len(at); mx = {"C": 4, "N": 3, "O": 2, "F": 1, "Cl": 1, "Br": 1, "S": 2}
        for i, j, o in bd:
            if i == j: return False
            used[i] += o; used[j] += o; deg[i] += 1; deg[j] += 1
        for i, a in enumerate(at):
            if a[3] is not None: continue
            if a[0] not in mx or (not a[1] and used[i] > mx[a[0]] + 1e-9) or (a[1] and deg[i] > 3): return False
        adj = {i: [] for i in range(len(at))}
        for i, j, o in bd:
            if o == 1.5: adj[i].append(j); adj[j].append(i)
        for i, a in enumerate(at):
            if a[1] and not (arom_cycle(i, adj, 6) or arom_cycle(i, adj, 5)): return False
        return True
    except Exception: return False
T = max(len(TOKEN.findall(s)) for s in smi_all) + 1; voc = Vocab(smi_all + [scaffold_smiles(n) for n in SCAF]); V = len(voc.itos); D = 48
Pm = np.array([props(s) for s in smi_all]); pmu, psd = Pm.mean(0), Pm.std(0); Cz = (Pm - pmu) / psd
ids_all = np.array([voc.enc(s, T) for s in smi_all]); sc_all = np.array([voc.enc(scaffold_smiles(n), T, eos=False) for n in fam])
perm = np.random.default_rng(0).permutation(len(smi_all)); tr_i, te_i = perm[:1400], perm[1400:]; train_set = set(smi_all[i] for i in tr_i)
print(f"vocabulary {V} tokens, sequences padded to {T}; {len(tr_i)} training and {len(te_i)} held-out molecules; one example: tokens {TOKEN.findall(smi_all[0])[:8]}..., scaffold {scaffold_smiles(fam[0])}")

In [ ]:
def init_params(seed=0, d=D, nprop=5):
    r = np.random.default_rng(seed); n = lambda *s, sc=0.1: r.normal(0, sc, s)
    return {"E": n(V, d), "P": n(T, d, sc=0.05), "S": n(V, d), "Wp": n(nprop, d, sc=0.2), "bp": np.zeros(d), "Wq": n(d, d, sc=d ** -0.5), "Wk": n(d, d, sc=d ** -0.5), "Wv": n(d, d, sc=d ** -0.5), "Wo": n(d, d, sc=d ** -0.5),
            "W1": n(d, 4 * d, sc=d ** -0.5), "b1": np.zeros(4 * d), "W2": n(4 * d, d, sc=(4 * d) ** -0.5), "b2": np.zeros(d), "Wout": n(d, V, sc=d ** -0.5), "bout": np.zeros(V)}
def forward(p, ids, sc_ids, c):
    B, T_ = ids.shape; d = p["E"].shape[1]; X = p["E"][ids] + p["P"][:T_] + p["S"][sc_ids] + (c @ p["Wp"] + p["bp"])[:, None, :]          # token + position + scaffold + property embeddings, summed
    Q, K, Vv = X @ p["Wq"], X @ p["Wk"], X @ p["Wv"]; S_ = Q @ K.transpose(0, 2, 1) / math.sqrt(d); S_ = np.where(np.triu(np.ones((T_, T_), bool), 1), -1e9, S_); A = softmax(S_)
    O = A @ Vv; X1 = X + O @ p["Wo"]; Hh = np.maximum(0, X1 @ p["W1"] + p["b1"]); X2 = X1 + Hh @ p["W2"] + p["b2"]
    return X2 @ p["Wout"] + p["bout"], dict(ids=ids, sc=sc_ids, c=c, X=X, Q=Q, K=K, V=Vv, A=A, O=O, X1=X1, H=Hh, X2=X2)
def loss_grads(p, ids_in, sc_ids, c, targets):
    logits, ca = forward(p, ids_in, sc_ids, c); B, T_, _ = logits.shape; mask = (targets != 0).astype(float); Pr = softmax(logits); idx = (np.arange(B)[:, None], np.arange(T_)[None, :], targets)
    loss = -(np.log(Pr[idx] + 1e-12) * mask).sum() / mask.sum(); dl = Pr.copy(); dl[idx] -= 1; dl *= mask[:, :, None] / mask.sum(); d = p["E"].shape[1]; g = {}
    g["Wout"] = np.einsum("btd,btv->dv", ca["X2"], dl); g["bout"] = dl.sum((0, 1)); dX2 = dl @ p["Wout"].T
    g["W2"] = np.einsum("bth,btd->hd", ca["H"], dX2); g["b2"] = dX2.sum((0, 1)); dH = (dX2 @ p["W2"].T) * (ca["H"] > 0); g["W1"] = np.einsum("btd,bth->dh", ca["X1"], dH); g["b1"] = dH.sum((0, 1)); dX1 = dX2 + dH @ p["W1"].T
    g["Wo"] = np.einsum("btd,bte->de", ca["O"], dX1); dO = dX1 @ p["Wo"].T; dA = dO @ ca["V"].transpose(0, 2, 1); dV = ca["A"].transpose(0, 2, 1) @ dO
    dS = ca["A"] * (dA - (dA * ca["A"]).sum(-1, keepdims=True)) / math.sqrt(d); dQ = dS @ ca["K"]; dK = dS.transpose(0, 2, 1) @ ca["Q"]
    g["Wq"], g["Wk"], g["Wv"] = (np.einsum("btd,bte->de", ca["X"], z_) for z_ in (dQ, dK, dV)); dX = dX1 + dQ @ p["Wq"].T + dK @ p["Wk"].T + dV @ p["Wv"].T
    g["E"] = np.zeros_like(p["E"]); np.add.at(g["E"], ca["ids"], dX); g["S"] = np.zeros_like(p["S"]); np.add.at(g["S"], ca["sc"], dX); g["P"] = np.zeros_like(p["P"]); g["P"][:ids_in.shape[1]] = dX.sum(0)
    dc = dX.sum(1); g["Wp"] = ca["c"].T @ dc; g["bp"] = dc.sum(0); return loss, g
p0 = init_params(); b_ = tr_i[:6]; inp_ = np.concatenate([np.ones((6, 1), int), ids_all[b_][:, :-1]], 1); l0, g0 = loss_grads(p0, inp_, sc_all[b_], Cz[b_], ids_all[b_]); worst = 0.0; r_ = np.random.default_rng(3)
for k in p0:
    for _ in range(3):
        ix = tuple(r_.integers(0, s) for s in p0[k].shape); a_, b2_ = ({q: z_.copy() for q, z_ in p0.items()} for _ in range(2)); a_[k][ix] += 1e-6; b2_[k][ix] -= 1e-6
        worst = max(worst, abs((loss_grads(a_, inp_, sc_all[b_], Cz[b_], ids_all[b_])[0] - loss_grads(b2_, inp_, sc_all[b_], Cz[b_], ids_all[b_])[0]) / 2e-6 - g0[k][ix]))
print(f"parameters: {sum(v.size for v in p0.values()):,} | gradient check on {3 * len(p0)} random weights of every array (embeddings, attention, feed-forward, output): largest difference {worst:.1e}"); assert worst < 1e-7

In [ ]:
def train(p, steps=1200, lr=3e-3, bs=48, seed=1):
    m = {k: np.zeros_like(v) for k, v in p.items()}; v = {k: np.zeros_like(x) for k, x in p.items()}; rg = np.random.default_rng(seed); hist = []
    for t in range(1, steps + 1):
        b = tr_i[rg.integers(0, len(tr_i), bs)]; inp = np.concatenate([np.ones((bs, 1), int), ids_all[b][:, :-1]], 1); l, g = loss_grads(p, inp, sc_all[b], Cz[b], ids_all[b]); hist.append(l)
        for k in p: m[k] = 0.9 * m[k] + 0.1 * g[k]; v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2; p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return hist
def generate(p, sc, c, seed=0, temp=1.0):
    """one SMILES per row: the scaffold tokens and the properties are fixed, the SMILES tokens are sampled one by one"""
    rg = np.random.default_rng(seed); B = len(c); ids = np.ones((B, 1), int); done = np.zeros(B, bool); out = [[] for _ in range(B)]
    for t in range(T):
        logits, _ = forward(p, ids, sc[:, :ids.shape[1]], c); pr = softmax(logits[:, -1] / temp); pr[:, :2] = 0; pr /= pr.sum(1, keepdims=True); nxt = np.array([rg.choice(V, p=pr[i]) for i in range(B)])
        for i in range(B):
            if not done[i]:
                if nxt[i] == 2: done[i] = True
                else: out[i].append(int(nxt[i]))
        ids = np.concatenate([ids, nxt[:, None]], 1)
        if done.all(): break
    return ["".join(voc.itos[i] for i in o) for o in out]
params = init_params(); t0 = time.perf_counter(); hist = train(params); print(f"trained in {time.perf_counter() - t0:.0f} s: loss {np.mean(hist[:20]):.3f} -> {np.mean(hist[-50:]):.3f}")
assert np.mean(hist[-50:]) < 0.2

In [ ]:
t0 = time.perf_counter(); n_g = 200; sel = te_i[:n_g]; g_cond = generate(params, sc_all[sel], Cz[sel], seed=1)
def evaluate(gen, sel_idx, label):
    val = np.array([valid(s) for s in gen]); vs = [s for s, v_ in zip(gen, val) if v_]; uniq = len(set(vs)) / max(1, len(vs)); nov = np.mean([s not in train_set for s in vs]) if vs else float("nan")
    keep = [(s, i) for s, i in zip(gen, sel_idx) if valid(s)]; ret = np.mean([scaffold_key(s) == scaffold_key(scaffold_smiles(fam[i])) for s, i in keep]) if keep else float("nan")
    exact = np.mean([s == smi_all[i] for s, i in zip(gen, sel_idx)]); mwerr = np.mean([abs(props(s)[0] - Pm[i, 0]) for s, i in keep]) if keep else float("nan")
    return dict(setting=label, validity=val.mean(), uniqueness=uniq, novelty=nov, scaffold_retention=ret, exact_reproduction=exact, MW_error=mwerr)
rows = [evaluate(g_cond, sel, "scaffold + properties of the held-out molecule")]
g_noscaf = generate(params, np.zeros_like(sc_all[sel]), Cz[sel], seed=2); rows.append(evaluate(g_noscaf, sel, "properties only (scaffold tokens set to pad)"))
g_noprop = generate(params, sc_all[sel], np.zeros_like(Cz[sel]), seed=3); rows.append(evaluate(g_noprop, sel, "scaffold only (all properties at the mean)"))
g_shuf = generate(params, sc_all[sel], Cz[np.random.default_rng(5).permutation(sel)], seed=4); rows.append(evaluate(g_shuf, sel, "scaffold + properties of ANOTHER molecule"))
ev = pd.DataFrame(rows); print(ev.round(3).to_string(index=False)); print(f"[{time.perf_counter() - t0:.0f} s]")
print("examples (target | generated):", [(smi_all[i], g) for i, g in zip(sel[:4], g_cond[:4])])
assert ev.validity[0] > 0.8 and ev.validity[0] > ev.validity[1:].max() + 0.2 and ev.scaffold_retention[0] > ev.scaffold_retention[1] + 0.1 and ev.MW_error[0] < ev.MW_error[3]

In [ ]:
# property control, as in the paper's Figure S14 in miniature: fix the molecular weight, draw the scaffold and the other properties at random from the training molecules
t0 = time.perf_counter(); rg = np.random.default_rng(7); rows = []; targets = [100, 140, 180, 220, 260]
for tmw in targets:
    j = rg.choice(tr_i, 150); c = Cz[j].copy(); c[:, 0] = (tmw - pmu[0]) / psd[0]; gens = generate(params, sc_all[j], c, seed=int(tmw)); vg = [s for s in gens if valid(s)]; mw = np.array([props(s)[0] for s in vg])
    rows.append((tmw, len(vg), mw.mean(), mw.std()))
pc = pd.DataFrame(rows, columns=["target MW", "valid of 150", "mean generated MW", "s.d."]); print(pc.round(1).to_string(index=False)); slope = np.polyfit(pc["target MW"], pc["mean generated MW"], 1)[0]
print(f"slope of the generated MW against the target MW: {slope:.2f} (1.0 = perfect control; the corpus has molecular weights from {Pm[:, 0].min():.0f} to {Pm[:, 0].max():.0f}, so the extremes cannot be reached) [{time.perf_counter() - t0:.0f} s]")
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.8)); ax[0].plot(hist, color=COLORS["blue"], lw=0.7); ax[0].set(xlabel="step", ylabel="cross-entropy per token", title="(a) training the miniature generator")
ax[1].errorbar(pc["target MW"], pc["mean generated MW"], yerr=pc["s.d."], fmt="o-", color=COLORS["orange"], capsize=3); ax[1].plot(targets, targets, "k--", lw=1, label="perfect control"); ax[1].set(xlabel="target molecular weight", ylabel="generated molecular weight", title="(b) property control"); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
assert slope > 0.4

## 7. The framework in miniature: generate, filter, rank
The complete loop: condition the generator on the scaffold and properties of an input molecule that is *active* in a filter; generate $500$ molecules; keep the valid and new ones whose predicted activity is below a threshold; rank the survivors by the standardized cosine similarity to the input. The paper's filter is a set of trained discriminative models. We have no hERG data here, so the filter is a **toy oracle**, written to follow only the qualitative medicinal chemistry that the paper cites (a basic aliphatic amine and lipophilic aromatic groups raise hERG affinity; polar groups lower it): $\mathrm{toy}=4.0+1.3\,[\text{basic amine}]+0.5\,(\text{aromatic atoms}/6)+0.3\,(\text{halogens})-0.4\,(\text{HBD})$. It is **not a model of hERG** and says nothing about any real compound; it lets us see what each step of the pipeline does.
**Stop and predict:** the input has a basic piperidine nitrogen inside its scaffold. What happens to the pass rate when the generator keeps the scaffold?

In [ ]:
def toy_pic50(smi):
    at, bd = parse_smiles(smi); H, nb_ = add_hydrogens(at, bd); n = len(at); el = [a[0] for a in at]; basic = 0
    for i in range(n):
        if el[i] == "N" and not at[i][1] and all(el[j] == "C" and not at[j][1] and all(o == 1 for k, o in nb_[j]) for j, o in nb_[i]): basic = 1
    return 4.0 + 1.3 * basic + 0.5 * sum(a[1] for a in at) / 6 + 0.3 * sum(e in ("F", "Cl", "Br") for e in el) - 0.4 * props(smi)[3]
inp = "c1cc(Cl)ccc1CCN1CCCCC1"; thr = 5.5; j_in = smi_all.index(inp) if inp in smi_all else None
sc_in = np.array([voc.enc(scaffold_smiles("phenethylpiperidine"), T, eos=False)]); c_in = ((props(inp) - pmu) / psd)[None, :]
print(f"input {inp} (in the training set: {inp in train_set}); toy score {toy_pic50(inp):.2f}; the filter keeps molecules with a toy score below {thr}")
t0 = time.perf_counter(); N_gen = 1000; gens = generate(params, np.repeat(sc_in, N_gen, 0), np.repeat(c_in, N_gen, 0), seed=11, temp=1.3)
vg = [s for s in dict.fromkeys(gens) if valid(s) and s != inp]; new = [s for s in vg if s not in train_set]; sc_ok = [s for s in vg if scaffold_key(s) == scaffold_key(scaffold_smiles("phenethylpiperidine"))]
score = {s: toy_pic50(s) for s in vg}; passed = [s for s in vg if score[s] < thr]
print(f"generated {N_gen} at the sampling temperature 1.3 (at 1.0 the model repeats a few molecules); unique, valid and different from the input {len(vg)}; not in the training set {len(new)}; with the input's scaffold {len(sc_ok)} ({100 * len(sc_ok) / max(1, len(vg)):.0f} %); passing the toy filter {len(passed)}; scaffold kept among those that pass: {sum(scaffold_key(s) == scaffold_key(scaffold_smiles('phenethylpiperidine')) for s in passed)} [{time.perf_counter() - t0:.0f} s]")
v_in = desc12(inp); sim = {s: cos(z(v_in), z(desc12(s))) for s in passed}; top = sorted(passed, key=lambda s: -sim[s])[:6]
print(pd.DataFrame([(s, round(score[s], 2), round(sim[s], 3), scaffold_key(s) == scaffold_key(scaffold_smiles("phenethylpiperidine"))) for s in top], columns=["candidate", "toy score", "cosine similarity", "input's scaffold"]).to_string(index=False))
print(f"toy scores of all valid generated molecules: mean {np.mean(list(score.values())):.2f}, range {min(score.values()):.2f} to {max(score.values()):.2f}; the input's score {toy_pic50(inp):.2f}")
fig, ax = plt.subplots(figsize=(6.4, 3.6)); ax.hist(list(score.values()), bins=np.arange(3.5, 7.0, 0.25), color=COLORS["blue"], alpha=0.8, label="generated (valid)"); ax.axvline(toy_pic50(inp), color=COLORS["red"], ls="--", label="input"); ax.axvline(thr, color="k", ls=":", label="filter")
ax.set(xlabel="toy score (NOT a hERG model)", ylabel="molecules", title="the filter acts on what the generator proposes"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
assert len(vg) > 30 and len(sc_ok) < len(vg) and all(valid(s) for s in passed)

## 8. Your turn
**Exercise 1.** Train without the scaffold embedding (set the scaffold tokens to the pad token during training). How do the scaffold retention and the validity change?
**Exercise 2.** Condition on a molecular weight larger than any in the corpus. What does the generator do?
**Exercise 3.** Change the threshold of the toy filter from $5.5$ to $5.0$ and to $6.0$. How many candidates survive, and what do they look like?
**Exercise 4.** In Section 2 use the metrics of another benchmark row of your choice and find its confusion matrix. Is it unique?
**Exercise 5.** Rank the candidates by the raw instead of the standardized cosine similarity. How does the top of the list change?

## 9. Self-check

In [ ]:
assert abs(fold_exp - 762) < 1 and (t2["integer solutions"] == 1).all() and cg[0] == (225, 36, 171, 42) and scaffold_key(pim) != scaffold_key(flu)
assert worst < 1e-7 and ev.validity[0] > 0.8 and raw.min() > 0.9 and abs(e.mean() + 0.386) < 5e-4
print("All checks passed: you reproduced the numbers in the post.")

# Part II. The real discriminative models, and checks against RDKit and scikit-learn
> **Read this first.** Part II does three things, each guarded. **(1)** With scikit-learn (preinstalled in Colab): our metrics, the mutual information and the cosine similarity against `f1_score`, `matthews_corrcoef`, `mutual_info_score` and `cosine_similarity`. **(2)** With RDKit (installed by the cell if it is missing): our parser, properties, Bemis--Murcko scaffolds and the validity check against RDKit's, including the SMILES strings that the miniature generator sampled. **(3)** The real CardioGenAI discriminative models: the cell clones the repository `gregory-kyro/CardioGenAI`, installs the packages that it can, calls the repository's `predict_cardiac_ion_channel_activity` for pimozide and fluspirilene (regression, hERG, CPU) and compares the predictions with the paper's ($7.629$ and $5.785$). The trained discriminative parameters are part of the repository according to its README (only the files of the generative transformer and the training data are hosted separately on Google Drive); if the installation or the call fails, the cell prints the reason. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
HAVE_SK = importlib.util.find_spec("sklearn") is not None; checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_SK:
    import warnings; warnings.filterwarnings("ignore")
    from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score, mutual_info_score
    from sklearn.metrics.pairwise import cosine_similarity
    try:
        ok_all = True
        for (ch, model), (n_, rep_) in T2.items():
            tp, fn, tn, fp = reconstruct(n_, rep_)[0]; yt = np.r_[np.ones(tp + fn), np.zeros(tn + fp)]; yp = np.r_[np.ones(tp), np.zeros(fn), np.ones(fp), np.zeros(tn)]
            ok_all &= abs(f1_score(yt, yp) * 100 - metrics6(tp, fn, tn, fp)[3]) < 1e-9 and abs(matthews_corrcoef(yt, yp) * 100 - metrics6(tp, fn, tn, fp)[5]) < 1e-9 and abs(accuracy_score(yt, yp) * 100 - metrics6(tp, fn, tn, fp)[0]) < 1e-9
        check("F1, MCC and accuracy of the reconstructed matrices of Table 2 equal scikit-learn's", ok_all)
        check("our mutual information (nats, from the binned descriptors) equals mutual_info_score", max(abs(MI[i, j] - mutual_info_score(bins[:, i], bins[:, j])) for i in range(12) for j in range(12)) < 1e-10)
        check("our cosine similarity equals cosine_similarity", abs(cos(z(v_in), z(Dm[5])) - cosine_similarity(z(v_in)[None], z(Dm[5])[None])[0, 0]) < 1e-12)
    except Exception as e_: print("stage could not be completed:", type(e_).__name__, e_)
else: print("scikit-learn is not available: install it and re-run this cell.")

In [ ]:
HAVE_RD = importlib.util.find_spec("rdkit") is not None
if not HAVE_RD and HAVE_SK:
    try:
        import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"]); HAVE_RD = importlib.util.find_spec("rdkit") is not None
    except Exception as e_: print("RDKit could not be installed:", type(e_).__name__, e_)
if HAVE_RD:
    try:
        from rdkit import Chem, rdBase
        from rdkit.Chem import Descriptors, rdMolDescriptors, Lipinski
        from rdkit.Chem.Scaffolds import MurckoScaffold
        print("RDKit", rdBase.rdkitVersion); test = [pim, flu] + smi_all[::60]; ok_mw = ok_ring = ok_scaf = True
        for s in test:
            m = Chem.MolFromSmiles(s); pr = props(s); ok_mw &= abs(Descriptors.MolWt(m) - pr[0]) < 0.02; ok_ring &= rdMolDescriptors.CalcNumRings(m) == pr[1] and Lipinski.NumHDonors(m) == pr[3] and rdMolDescriptors.CalcNumHeteroatoms(m) == pr[4]
            sm_ = Chem.MolFromSmiles(MurckoScaffold.MurckoScaffoldSmiles(mol=m)); ok_scaf &= sm_.GetNumAtoms() == len(murcko_atoms(s)[2])
        check("molecular weight, rings, hydrogen-bond donors and heteroatoms equal RDKit's (pimozide, fluspirilene and 27 corpus molecules)", ok_mw and ok_ring)
        check("the number of atoms of our Bemis-Murcko scaffold equals RDKit's for the same molecules", ok_scaf)
        rot_diff = {nm: (rdMolDescriptors.CalcNumRotatableBonds(Chem.MolFromSmiles(s)), int(props(s)[2])) for nm, s in (("pimozide", pim), ("fluspirilene", flu))}; print("rotatable bonds, RDKit against ours (RDKit's definition is stricter):", rot_diff)
        a_ = [valid(s) for s in g_cond + g_noscaf]; b_ = [Chem.MolFromSmiles(s) is not None for s in g_cond + g_noscaf]; print(f"validity of the {len(a_)} sampled strings: ours {np.mean(a_):.3f}, RDKit {np.mean(b_):.3f}; strings we call valid that RDKit rejects: {sum(x and not y for x, y in zip(a_, b_))}")
        check("every string that our check accepts is read by RDKit (our check is not more permissive)", all(y for x, y in zip(a_, b_) if x))
        check("Murcko scaffolds: pimozide and fluspirilene have different scaffolds in RDKit too", MurckoScaffold.MurckoScaffoldSmiles(pim) != MurckoScaffold.MurckoScaffoldSmiles(flu))
    except Exception as e_: print("RDKit stage could not be completed:", type(e_).__name__, e_)
else: print("RDKit is not available: the RDKit checks were skipped.")

In [ ]:
import subprocess, re, glob
REPO = "CardioGenAI"
def run_real_models():
    if not os.path.isdir(REPO):
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/gregory-kyro/CardioGenAI.git", REPO])
    print("repository files:", sorted(os.listdir(REPO))[:20]); env = os.path.join(REPO, "environment.yml")
    if os.path.isfile(env): print("environment.yml (first lines):", open(env).read().splitlines()[:12])
    for pkg in ("torch_geometric", "h5py", "pandas", "scikit-learn"):
        if importlib.util.find_spec(pkg.replace("-", "_").replace("scikit_learn", "sklearn")) is None:
            try: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            except Exception as e_: print("could not install", pkg, type(e_).__name__)
    sys.path.insert(0, os.path.abspath(REPO)); from src.Discriminator import predict_cardiac_ion_channel_activity
    out = {}
    for nm, s in (("pimozide", pim), ("fluspirilene", flu)):
        res = predict_cardiac_ion_channel_activity(s, "regression", True, False, False, "cpu"); out[nm] = res; print(nm, "->", res)
    return out
real = None
if os.environ.get("CARDIOGENAI_SKIP") != "1" and HAVE_RD:
    try: real = run_real_models()
    except Exception as e_: print("the real models could not be run here:", type(e_).__name__, str(e_)[:300])
else: print("skipped (RDKit is needed; set CARDIOGENAI_SKIP=0 or unset it to run).")
def first_float(x):
    nums = [float(v) for v in re.findall(r"-?\d+\.\d+", str(x))]; return nums[0] if nums else None
if real is not None:
    pm, fl = first_float(real["pimozide"]), first_float(real["fluspirilene"]); print("extracted predictions:", pm, fl, "| the paper: 7.629 and 5.785")
    check("the repository's hERG regression model reproduces the paper's prediction for pimozide (7.629, tolerance 0.05)", pm is not None and abs(pm - 7.629) < 0.05)
    check("... and for fluspirilene (5.785, tolerance 0.05)", fl is not None and abs(fl - 5.785) < 0.05)
if checks:
    ok = [lab for lab, p_ in checks if p_]; bad = [lab for lab, p_ in checks if not p_]; print(f"\nPart II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)

### What to look for
* **The repository call.** The README documents the function and its arguments; what it returns (a number, a table, printed lines) is shown by the cell, and the first number found is compared with the paper's. If the two predictions are reproduced, the discriminative models of the paper are working in your session. Because pimozide is in the training set of the model (Table 3 of the paper), agreement for pimozide is not evidence of generalization.
* **Our validity check against RDKit.** A string that our check accepts but RDKit rejects would be a hole in our check (aromaticity, for instance); the cell counts them.
* **Rotatable bonds.** RDKit's default definition is stricter than ours, and the cell prints both counts.